In [18]:
# Notebook 02b: Logistic Regression Tuning

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score
from sklearn.ensemble import RandomForestClassifier
import joblib
import warnings

from sklearn.utils import class_weight

warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('../data/spy_features.csv', index_col='Date', parse_dates=True)

# Check the shape and columns
print(f"Data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

Data shape: (2939, 25)
Columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'ATR', 'OBV', 'Stoch_K', 'Stoch_D', 'Williams_R', 'ROC', 'Label', 'Daily_Return']

First few rows:
                 Close        High         Low        Open     Volume  \
Date                                                                    
2014-10-16  153.016907  154.093042  150.240304  150.379953  270391000   
2014-10-17  154.824188  155.875681  154.125926  154.783112  214625000   
2014-10-20  156.327454  156.450670  154.495559  154.544846  130011000   
2014-10-21  159.424438  159.531222  157.296798  157.461092  154949000   
2014-10-22  158.290756  160.114440  158.225036  159.703701  151822000   

                  RSI      MACD  MACD_Signal  MACD_Hist    BB_Upper  ...  \
Date                                                                 ...   
2014-10-16  29.535651 -2.301859

In [3]:
# drop features
cols_to_drop = ['High', 'Low', 'Open', 'BB_Middle', 'BB_Lower', 'SMA_20', 'SMA_200', 'EMA_12', 'EMA_26']

df = df.drop(columns=cols_to_drop)

df = df.dropna()



KeyError: "['High', 'Low', 'Open', 'BB_Middle', 'BB_Lower', 'SMA_20', 'SMA_200', 'EMA_12', 'EMA_26'] not found in axis"

In [4]:
# define X and Y
X = df.drop(columns=['Label'])
y = df['Label']


print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features: {X.columns.tolist()}")
print(f"Label distribution:\n{y.value_counts()}")

X shape: (2938, 15)
y shape: (2938,)
Features: ['Close', 'Volume', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'SMA_50', 'ATR', 'OBV', 'Stoch_K', 'Stoch_D', 'Williams_R', 'ROC', 'Daily_Return']
Label distribution:
Label
1    1614
0    1324
Name: count, dtype: int64


In [5]:
tscv = TimeSeriesSplit(n_splits=5)

# scaler, will be fitted inside each fold
scaler = StandardScaler()

# preview the fold sizes
print("Walk-forward fold sizes:")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
    print(f"Fold {fold}: Train={len(train_idx)} rows, Validate={len(val_idx)} rows")

Walk-forward fold sizes:
Fold 1: Train=493 rows, Validate=489 rows
Fold 2: Train=982 rows, Validate=489 rows
Fold 3: Train=1471 rows, Validate=489 rows
Fold 4: Train=1960 rows, Validate=489 rows
Fold 5: Train=2449 rows, Validate=489 rows


In [28]:
for i, (train_idx, val_idx) in enumerate(tscv.split(X)):
    print(f"Fold {i+1}: train {X.index[train_idx[0]]} to {X.index[train_idx[-1]]}, "
          f"val {X.index[val_idx[0]]} to {X.index[val_idx[-1]]}")

Fold 1: train 2014-10-17 00:00:00 to 2016-09-30 00:00:00, val 2016-10-03 00:00:00 to 2018-09-11 00:00:00
Fold 2: train 2014-10-17 00:00:00 to 2018-09-11 00:00:00, val 2018-09-12 00:00:00 to 2020-08-20 00:00:00
Fold 3: train 2014-10-17 00:00:00 to 2020-08-20 00:00:00, val 2020-08-21 00:00:00 to 2022-08-01 00:00:00
Fold 4: train 2014-10-17 00:00:00 to 2022-08-01 00:00:00, val 2022-08-02 00:00:00 to 2024-07-12 00:00:00
Fold 5: train 2014-10-17 00:00:00 to 2024-07-12 00:00:00, val 2024-07-15 00:00:00 to 2026-06-25 00:00:00


In [16]:
c_values = [0.01, 0.1, 1, 10, 100]
class_weights = ['balanced', None]
lr_fold_metrics = []

# Outer loop
for C in c_values:
    for cw in class_weights:
        for fold_num, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_val_scaled = scaler.transform(X_val)
            model = LogisticRegression(C=C, max_iter=1000, random_state=42, class_weight=cw)
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_val_scaled)

            #calculate and append metrics
            metrics = {
            'C': C,
            'class_weight': cw,
            'fold': fold_num,
            'accuracy': accuracy_score(y_val, y_pred),
            'precision': precision_score(y_val, y_pred),
            'recall': recall_score(y_val, y_pred),
            'f1': f1_score(y_val, y_pred),
            'precision_down': precision_score(y_val, y_pred, pos_label=0),
            'recall_down': recall_score(y_val, y_pred, pos_label=0),
            'precision_up': precision_score(y_val, y_pred, pos_label=1),
            'recall_up': recall_score(y_val, y_pred, pos_label=1)
            }
            lr_fold_metrics.append(metrics)
            print(f"Fold {fold_num}: Accuracy={metrics['accuracy']:.3f}, Precision={metrics['precision']:.3f}: Recall={metrics['recall']:.3f}, F1={metrics['f1']:.3f}")
df = pd.DataFrame(lr_fold_metrics)

print("\n--- Logistic Regression Average by C ---")
grouped = df.groupby(['C', 'class_weight'], dropna=False)[['accuracy', 'precision', 'recall', 'f1', 'precision_down', 'recall_down', 'precision_up', 'recall_up']].mean()
print(grouped)




Fold 1: Accuracy=0.431, Precision=0.400: Recall=0.007, F1=0.014
Fold 2: Accuracy=0.552, Precision=0.580: Recall=0.781, F1=0.666
Fold 3: Accuracy=0.538, Precision=0.538: Recall=0.992, F1=0.698
Fold 4: Accuracy=0.448, Precision=0.486: Recall=0.531, F1=0.507
Fold 5: Accuracy=0.499, Precision=0.546: Recall=0.670, F1=0.602
Fold 1: Accuracy=0.438, Precision=0.625: Recall=0.018, F1=0.035
Fold 2: Accuracy=0.575, Precision=0.574: Recall=0.982, F1=0.725
Fold 3: Accuracy=0.538, Precision=0.538: Recall=1.000, F1=0.699
Fold 4: Accuracy=0.538, Precision=0.537: Recall=0.985, F1=0.695
Fold 5: Accuracy=0.560, Precision=0.564: Recall=0.975, F1=0.714
Fold 1: Accuracy=0.431, Precision=0.400: Recall=0.007, F1=0.014
Fold 2: Accuracy=0.497, Precision=0.554: Recall=0.609, F1=0.580
Fold 3: Accuracy=0.536, Precision=0.537: Recall=0.992, F1=0.697
Fold 4: Accuracy=0.479, Precision=0.510: Recall=0.676, F1=0.581
Fold 5: Accuracy=0.505, Precision=0.547: Recall=0.710, F1=0.618
Fold 1: Accuracy=0.436, Precision=0.556:

In [19]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'class_weight': ['balanced', None]
}
recall_down_scorer = make_scorer(recall_score, pos_label=0)

rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    scoring={'recall_down': recall_down_scorer, 'accuracy': 'accuracy'},
    refit='recall_down',
    verbose = 3
)

rf_grid_search.fit(X, y)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END class_weight=balanced, max_depth=5, n_estimators=100; accuracy: (test=0.472) recall_down: (test=0.877) total time=   0.2s
[CV 2/5] END class_weight=balanced, max_depth=5, n_estimators=100; accuracy: (test=0.444) recall_down: (test=0.543) total time=   0.2s
[CV 3/5] END class_weight=balanced, max_depth=5, n_estimators=100; accuracy: (test=0.474) recall_down: (test=0.942) total time=   0.4s
[CV 4/5] END class_weight=balanced, max_depth=5, n_estimators=100; accuracy: (test=0.472) recall_down: (test=0.529) total time=   0.3s
[CV 5/5] END class_weight=balanced, max_depth=5, n_estimators=100; accuracy: (test=0.558) recall_down: (test=0.009) total time=   0.5s
[CV 1/5] END class_weight=balanced, max_depth=5, n_estimators=200; accuracy: (test=0.472) recall_down: (test=0.863) total time=   0.4s
[CV 2/5] END class_weight=balanced, max_depth=5, n_estimators=200; accuracy: (test=0.450) recall_down: (test=0.557) total time=  

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'max_depth': [5, 10, ...], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'recall_down': make_scorer(r..., pos_label=0)}"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'recall_down'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- 

In [20]:
print(rf_grid_search.best_params_)

{'class_weight': None, 'max_depth': None, 'n_estimators': 200}


In [21]:
rf_grid_search.best_score_

np.float64(0.6172360000944106)

In [22]:
best_index = rf_grid_search.best_index_
best_accuracy = rf_grid_search.cv_results_['mean_test_accuracy'][best_index]
print(f"Accuracy for the winning combination: {best_accuracy:.3f}")

Accuracy for the winning combination: 0.488


In [23]:
rf_results = pd.DataFrame(rf_grid_search.cv_results_)
rf_results_display = rf_results[['param_n_estimators', 'param_max_depth', 'param_class_weight',
                                   'mean_test_recall_down', 'mean_test_accuracy']]
rf_results_display = rf_results_display.sort_values('mean_test_recall_down', ascending=False)
print(rf_results_display)

    param_n_estimators param_max_depth param_class_weight  \
11                 200            None                NaN   
10                 100            None                NaN   
1                  200               5           balanced   
0                  100               5           balanced   
5                  200            None           balanced   
4                  100            None           balanced   
8                  100              10                NaN   
9                  200              10                NaN   
3                  200              10           balanced   
2                  100              10           balanced   
6                  100               5                NaN   
7                  200               5                NaN   

    mean_test_recall_down  mean_test_accuracy  
11               0.617236            0.488344  
10               0.614831            0.491207  
1                0.585239            0.478528  
0             

In [24]:
from sklearn.ensemble import GradientBoostingClassifier

gb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1]
}

gb_grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=gb_param_grid,
    cv=tscv,
    scoring={'recall_down': recall_down_scorer, 'accuracy': 'accuracy'},
    refit='recall_down',
    verbose=3
)

gb_grid_search.fit(X, y)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV 1/5] END learning_rate=0.01, max_depth=3, n_estimators=100; accuracy: (test=0.454) recall_down: (test=0.816) total time=   0.2s
[CV 2/5] END learning_rate=0.01, max_depth=3, n_estimators=100; accuracy: (test=0.550) recall_down: (test=0.319) total time=   0.7s
[CV 3/5] END learning_rate=0.01, max_depth=3, n_estimators=100; accuracy: (test=0.462) recall_down: (test=1.000) total time=   0.7s
[CV 4/5] END learning_rate=0.01, max_depth=3, n_estimators=100; accuracy: (test=0.491) recall_down: (test=0.220) total time=   1.5s
[CV 5/5] END learning_rate=0.01, max_depth=3, n_estimators=100; accuracy: (test=0.542) recall_down: (test=0.117) total time=   2.1s
[CV 1/5] END learning_rate=0.01, max_depth=3, n_estimators=200; accuracy: (test=0.446) recall_down: (test=0.882) total time=   0.7s
[CV 2/5] END learning_rate=0.01, max_depth=3, n_estimators=200; accuracy: (test=0.513) recall_down: (test=0.419) total time=   1.3s
[CV 3/5] END lea

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoost...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1], 'max_depth': [3, 5], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'recall_down': make_scorer(r..., pos_label=0)}"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'recall_down'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : comput

In [25]:
gb_results = pd.DataFrame(gb_grid_search.cv_results_)
gb_results_display = gb_results[['param_n_estimators', 'param_max_depth', 'param_learning_rate',
                                   'mean_test_recall_down', 'mean_test_accuracy']]
gb_results_display = gb_results_display.sort_values('mean_test_recall_down', ascending=False)
print(gb_results_display)

   param_n_estimators  param_max_depth  param_learning_rate  \
5                 200                3                 0.10   
7                 200                5                 0.10   
4                 100                3                 0.10   
6                 100                5                 0.10   
3                 200                5                 0.01   
1                 200                3                 0.01   
2                 100                5                 0.01   
0                 100                3                 0.01   

   mean_test_recall_down  mean_test_accuracy  
5               0.681960            0.479755  
7               0.647778            0.484254  
4               0.644642            0.493661  
6               0.621533            0.481391  
3               0.573793            0.489162  
1               0.525224            0.488344  
2               0.520168            0.498569  
0               0.494544            0.499796  


In [26]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_param_grid = {
    'max_iter': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'class_weight': ['balanced', None]
}

hgb_grid_search = GridSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=42),
    param_grid=hgb_param_grid,
    cv=tscv,
    scoring={'recall_down': recall_down_scorer, 'accuracy': 'accuracy'},
    refit='recall_down',
    verbose=3
)

hgb_grid_search.fit(X, y)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV 1/5] END class_weight=balanced, learning_rate=0.01, max_depth=3, max_iter=100; accuracy: (test=0.476) recall_down: (test=0.858) total time=   3.3s
[CV 2/5] END class_weight=balanced, learning_rate=0.01, max_depth=3, max_iter=100; accuracy: (test=0.442) recall_down: (test=0.571) total time=   0.7s
[CV 3/5] END class_weight=balanced, learning_rate=0.01, max_depth=3, max_iter=100; accuracy: (test=0.511) recall_down: (test=0.690) total time=   0.7s
[CV 4/5] END class_weight=balanced, learning_rate=0.01, max_depth=3, max_iter=100; accuracy: (test=0.442) recall_down: (test=0.361) total time=   0.7s
[CV 5/5] END class_weight=balanced, learning_rate=0.01, max_depth=3, max_iter=100; accuracy: (test=0.564) recall_down: (test=0.000) total time=   0.7s
[CV 1/5] END class_weight=balanced, learning_rate=0.01, max_depth=3, max_iter=200; accuracy: (test=0.464) recall_down: (test=0.877) total time=   0.6s
[CV 2/5] END class_weight=balance

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 5], 'max_iter': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'recall_down': make_scorer(r..., pos_label=0)}"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'recall_down'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the tot

In [27]:
hgb_grid_search.cv_results_

{'mean_fit_time': array([1.29970789, 0.80429158, 0.77337871, 0.82956309, 0.77192249,
        0.77875714, 0.79024067, 0.8182879 , 0.04506965, 0.21227045,
        0.07225862, 0.12634859, 0.044768  , 0.07813225, 0.06550279,
        0.12060747]),
 'std_fit_time': array([1.04459234, 0.04709167, 0.03633579, 0.05115321, 0.04926621,
        0.0402326 , 0.04187667, 0.04987573, 0.00661995, 0.28367294,
        0.00863803, 0.01897475, 0.00889368, 0.00681082, 0.0073813 ,
        0.02366443]),
 'mean_score_time': array([0.00440316, 0.00539956, 0.00470176, 0.00524549, 0.00440812,
        0.00540009, 0.00459971, 0.00519981, 0.00379987, 0.0058136 ,
        0.00422006, 0.00519977, 0.00454459, 0.00480003, 0.0041997 ,
        0.00579977]),
 'std_score_time': array([0.00049376, 0.00080009, 0.00039906, 0.00047823, 0.00048368,
        0.00049006, 0.00079985, 0.00039999, 0.00039992, 0.00159345,
        0.00077081, 0.00039942, 0.0009835 , 0.00040016, 0.00074834,
        0.00116642]),
 'param_class_weight': mas

No model across four families and hyperparameter combinations beat the 55% "always up" baseline accuracy (best: Logistic Regression at 0.530). Equity curve and Sharpe ratio will be the deciding factor going forward, not raw accuracy.